In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- design_matrix_model ---
class QStandardItem:
    def __init__(self, value):
        self.value = value
    def __str__(self):
        return self.value
    def __repr__(self):
        return f"QStandardItem({self.value!r})"

class RecordingModel:
    def __init__(self):
        self.rows = []
        self.headers = []
    def appendRow(self, row):
        self.rows.append([str(item) for item in row])
    def setVerticalHeaderLabels(self, labels):
        self.headers = list(labels)

FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PD = pd.DataFrame(
    {"id": [1, 2, 3], "value": [10, 20, 30]},
    index=pd.Index(["run_a", "run_b", "run_c"], name="realization"),
)
# The reference migration keeps the pandas index's identity as an explicit
# "realization" column (there being no polars index) - the previous fixture
# omitted it entirely, so no condition's generated code could possibly have
# reproduced the per-run header labels regardless of translation quality.
FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PL = pl.DataFrame({"id": [1, 2, 3], "value": [10, 20, 30], "realization": ["run_a", "run_b", "run_c"]})
FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF = FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PL
FIX_DESIGN_MATRIX_MODEL_MODEL = RecordingModel()

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_design_matrix_model(design_matrix_df, model):
    header_labels = design_matrix_df.columns.astype(str).tolist()
    for index, _ in design_matrix_df.iterrows():
        model.appendRow([
            QStandardItem(str(design_matrix_df.at[index, col]))
            for col in design_matrix_df.columns
        ])
    model.setVerticalHeaderLabels(design_matrix_df.index.astype(str).tolist())
    return header_labels

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_design_matrix_model(design_matrix_df, model):
    header_labels = [str(col) for col in design_matrix_df.columns]
    for row in design_matrix_df.iter_rows():
        model.appendRow([
            QStandardItem(str(value))
            for value in row
        ])
    model.setVerticalHeaderLabels([str(i) for i in range(design_matrix_df.height)])
    return header_labels

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: design_matrix_model ===

def _design_matrix_state(model):
    return {"rows": model.rows, "headers": model.headers}

# L1 smoke – generated
try:
    _model = RecordingModel()
    _r = gen_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PL, _model)
    print("✅ L1 smoke gen_design_matrix_model: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_design_matrix_model: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _model = RecordingModel()
    _rb = before_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PD, _model)
    print("✅ L1 smoke before_design_matrix_model: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_design_matrix_model: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _before_model = RecordingModel()
    _gen_model = RecordingModel()
    _rb = before_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PD, _before_model)
    _rg = gen_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PL, _gen_model)
    if list(_rb) == list(_rg) and _before_model.rows == _gen_model.rows and _before_model.headers == _gen_model.headers:
        print("✅ L2 equivalence design_matrix_model: MATCH")
    else:
        print(f"❌ L2 equivalence design_matrix_model: MISMATCH — before={_design_matrix_state(_before_model)}, gen={_design_matrix_state(_gen_model)}")
except Exception as _e:
    print(f"❌ L2 equivalence design_matrix_model: setup error — {type(_e).__name__}: {_e}")

# L3 edge – empty DataFrame input
try:
    _before_model = RecordingModel()
    _gen_model = RecordingModel()
    _rb = before_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PD.head(0), _before_model)
    _rg = gen_design_matrix_model(FIX_DESIGN_MATRIX_MODEL_DESIGN_MATRIX_DF_PL.head(0), _gen_model)
    if list(_rb) == list(_rg) and _before_model.rows == _gen_model.rows and _before_model.headers == _gen_model.headers:
        print("✅ L3 edge design_matrix_model empty: MATCH")
    else:
        print(f"❌ L3 edge design_matrix_model empty: MISMATCH — before={_design_matrix_state(_before_model)}, gen={_design_matrix_state(_gen_model)}")
except Exception as _e:
    print(f"❌ L3 edge design_matrix_model: {type(_e).__name__}: {_e}")
